# Post-Cleaning
1. Remove non-music items (based on audio classification with [PANN](https://github.com/qiuqiangkong/audioset_tagging_cnn)
2. De-duplication by file hashes
3. De-duplication by audio fingerprinting


## Load dataset

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd


def load_dataset(path):
    """
    Load dataset from the specified path.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at {path}")
    
    def inverse_split_dict(split_dict):
        # Add split column to dvi2Torch based on dvi2_torch["split"]
        clique_to_split = {}
        for split_name, sub_dict in split_dict.items():
            for clique in sub_dict.keys():
                clique_to_split[clique] = split_name
        return clique_to_split

    if path.endswith(".json"):
        with open(path, "r") as f:
            meta = json.load(f)
    elif path.endswith(".pt"):
        meta = torch.load(path, weights_only=False)
    if isinstance(meta, dict) and "info" in meta:
        info =  meta["info"]
        split = meta["split"]
    else:
        info, split = meta

    df = pd.DataFrame.from_dict(info, orient="index")
           
    clique2split = inverse_split_dict(split)
    df["split"] = df["clique"].map(clique2split)
    
    if "youtube_id" not in df.columns:
        df["youtube_id"] = df.filename.apply(lambda x: x.split("/")[-1].split(".")[0])
    
    df["dvi"] = ~df.apply(lambda x: x.youtube_id in x.version, axis=1)
    
    return df, meta    

def print_df_summary(df, subset="overall"):
    if subset != "overall":
        df = df.query(f"split == '{subset}'")
    print(f"Summary statistics for {subset}")

    # Column stats
    stats = []
    for col in ["clique", "version", "youtube_id"]:
        unique_count = df[col].nunique()
        total_count = df[col].count()
        stats.append(f"{col:<12} | Unique: {unique_count:>7,} | Total: {total_count:>7,}")
    print("\n".join(stats))

    # DVI distribution
    if "dvi" in df.columns:
        n_true = df["dvi"].sum()
        n_false = (~df["dvi"]).sum()
        total = len(df)
        print("\nDVI distribution:")
        print(f"  True:  {n_true:>7,} ({n_true/total:5.2%})")
        print(f"  False: {n_false:>7,} ({n_false/total:5.2%})")

    # Version per clique stats
    version_per_clique = df.groupby("clique")["version"].nunique()
    print("\nVersion per clique:")
    print(f"  Min: {version_per_clique.min():>3}  | Max: {version_per_clique.max():>3}  | "
          f"Mean: {version_per_clique.mean():>5.2f}  | Median: {version_per_clique.median():>5.2f}  | "
          f"Std: {version_per_clique.std():>5.2f}")
    print("=" * 40)

df, meta = load_dataset("../data/final/divers.pt")
print_df_summary(df, "train")


Summary statistics for train
clique       | Unique:  78,640 | Total: 1,007,643
version      | Unique: 1,007,643 | Total: 1,007,643
youtube_id   | Unique: 1,007,603 | Total: 1,007,643

DVI distribution:
  True:  328,117 (32.56%)
  False: 679,526 (67.44%)

Version per clique:
  Min:   2  | Max: 470  | Mean: 12.81  | Median:  5.00  | Std: 18.38


## Remove non-musical items

In [12]:
tagging_output_dir = "data/tagging_output_nonmusic_txt"

# List to store filenames without extensions
filenames_no_ext = []

for root, dirs, files in os.walk(tagging_output_dir):
    for file in files:
        # Remove the extension
        name_without_ext = os.path.splitext(file)[0]
        filenames_no_ext.append(name_without_ext)

print(filenames_no_ext)
print(f"Removing {len(filenames_no_ext)} entries from the dataset.")
df = df[~df['youtube_id'].isin(filenames_no_ext)]


['S_JOXo46hi4', 'xTycYNeCDGc', 'IJzzZeYlMhw', 'ufxaG_ursPo', 'Y24crLnKi0A', 'zTxt9ulD1VI', 'AgdZTljXc5s', 'HXJlD_ubBMc', 'O4x4RmeBmo8', 'mverLtNDVxo', 'mNHCVGaGVqg', 'uYHCrxvUFlQ', 'nPlv3jPIQww', 'MeMXd1b6Yr8', 'LKPaRBKqLcs', 'x7JzJAIVOhc', '3nFCZG39vME', 'WLVqYyUmwbY', 'EAn9SgAq1Xs', 'bICrtVM7GLo', '8J5_61-Wm04', 'TO8CdwCFDs8', 'sGQYPElgvdg', 'TCIMxzNgLn0', 'Vebex4eTu9M', 'bD8O3yZxvQc', 'ZuzY9RDS0Dw', 'wzW3A01rBas', 'k8q-rzAvIP4', 'eMnIS_HDZhI', 'Vm7-X1M6_zo', '22Bx3tSBAEk', 'mmZqffzxTmw', 'dXb9N-UHNhQ', 'kBWSGO5kfSk', 'jOdBFEhZpT8', 'f09tV0VHY-k', 'w4E2kaVPw3Y', '1EvLaKoEKXY', 'Y1omU6vCuA0', 'S4nOJxc_JQ4', 'A-p7Z2emFN0', 'ob--gZVHt_w', 'S92Wh7-ZKUg', 'ZkOKV5u2nBg', 'MGpUxumgZdE', 'GTiJN2Tuxw0', 'jPWr6dlNMic', 'Fc3Wkh8Ht08', 'rm0YeGtn0fw', 'hRztQuK72iE', 'gSYeQuJvsIE', 'h93FOTIvJTc', 'Kg207Rt032c', 'n8A1PgsYmI8', 'A8qSg1-NHP8', 'EGA3bIoQ0ZE', '6h0A96D6ORI', 'Q8pDlN03ZAU', '1U5D-GI9rTM', 'phiPeZKqrWQ', 'Glv9aqAv3ZE', 'NgvXTgc8b_0', 'ilIgYNnusOE', 'vyhaUzzg6Zw', 'B4Lq6HPuvzY', 'Iq5aUwTD

## Deduplication
### By Hashes

In [2]:
from collections import defaultdict


def parse_hashed_file(file_path):
    """
    Parses a file containing hashes and associated file paths.
    
    Args:
        file_path (str): Path to the input file.
    """
    # Dictionary to store results
    hash_to_files = defaultdict(list)

    with open(file_path, "r") as f:
        current_hash = None
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines
            # Accept both "Hash:" and "PCM Hash:" (case-insensitive)
            key, _, value = line.partition(':')
            if key and key.strip().lower() in ("hash", "pcm hash"):
                current_hash = value.strip()
                continue
            elif current_hash is not None:
                # Add file path to the current hash
                hash_to_files[current_hash].append(line)

    # Convert to normal dict if you like
    return dict(hash_to_files)

def get_hashed_inverted(file_path):
    hash_to_files = parse_hashed_file(file_path)
    # Invert hash_to_files: map filename -> hash
    filename_to_hash = {}
    for h, paths in hash_to_files.items():
        for p in paths:
            filename = os.path.splitext(os.path.basename(p))[0]
            filename_to_hash[filename] = h
    return filename_to_hash


# Path to your input file
file_path_basic = "/data/sha/audio/hashed_basic.txt"

hashed = get_hashed_inverted(file_path_basic)

# Now map youtube_id to hash
df['hash'] = df['youtube_id'].map(hashed)
print(f"Number of duplicates based on hash: {df.hash.dropna().duplicated().sum():,}")
print(f"Number of duplicates based on hash: {df.hash.dropna().duplicated().sum() / len(df):.2%}")


Number of duplicates based on hash: 40,177
Number of duplicates based on hash: 2.89%


#### Conflicting duplicates
Duplicates across different cliques.

In [11]:
df_hash = df.dropna(subset=['hash'])

# get conflicting hashes
grouped = df_hash.groupby('hash')
hashes_conflicting = grouped['clique'].nunique()
hashes_conflicting = hashes_conflicting[hashes_conflicting > 1].index

# filter df
df_conflicting = df_hash[df_hash['hash'].isin(hashes_conflicting)]
df_conflicting = df_conflicting.sort_values(['hash', 'clique'])

# get clique level metadata
df_clique_summary = df.groupby('clique', as_index=False).agg({
    'artist': 'first',  # first non-null in group
    'title': 'first'
})

# merge
cols = ["clique", "version", "youtube_id", "hash"]
df_conflicting = pd.merge(
    df_conflicting[cols].sort_values(['hash', 'clique']),
    df_clique_summary,
    on="clique",
    how="left"
)
df_conflicting

,clique,version,youtube_id,hash,artist,title
0,C-0092182,9McuWeohHt0,9McuWeohHt0,0044e89015faca56a8a36417a173cb840307fbfb72724b...,The Spotnicks,Harmour Love
1,C-0109215,cV4SjQSxIOk,cV4SjQSxIOk,0044e89015faca56a8a36417a173cb840307fbfb72724b...,Aretha Franklin,Ain't Nobody Ever Loved You
2,C-0227930,QaDqUdftBxg,QaDqUdftBxg,0044e89015faca56a8a36417a173cb840307fbfb72724b...,Anne Wylie,The Bonny Swans
3,C-0051631,Uk5B9h8GjI8,Uk5B9h8GjI8,00e3d7299e94f59a2f055eba223361f10067ce250a0d57...,The Pointer Sisters,All I Know Is The Way I Feel
4,C-0065707,HdL5eQSrw4Y,HdL5eQSrw4Y,00e3d7299e94f59a2f055eba223361f10067ce250a0d57...,Earth Crisis,The Wanton Song
...,...,...,...,...,...,...
1989,C-0112013,gFghvJj9F_Q,gFghvJj9F_Q,fea6ccfbcf6680f21049ec22932ff34148f7f6299b8971...,Erik & Marion,Itsy Bitsy Teenie Weenie Yellow Polka Dot Bikini
1990,C-0061050,ThVjTRgTG54,ThVjTRgTG54,fea95fa8fb7619391490defae71b72c244892681bb0334...,Charlie Byrd,Where's The Playground Susie?
1991,C-0118436,5etlUeR_HxI,5etlUeR_HxI,fea95fa8fb7619391490defae71b72c244892681bb0334...,Frank Sinatra,Something Wonderful Happens In Summer
1992,C-0089057,V-0911337,ujf6M4xt3FQ,ff7bb2288e1e28fc0c4b40dbe7dd77128d2e2d1e2a249a...,Bo Diddley,Cadillac


#### Deduplication Strategy
We delete by the following strategy: If a hash occurrs across cliques, drop all the versions with that hash. If a hash only occurs in one clique, prioritize an item which is from Discogs (dvi is True). If none is from Discogs, just pick the first to keep and drop the rest of the duplicates by hash. 


In [13]:
# split df
df_no_hash = df[df['hash'].isna()]
df_with_hash = df[df['hash'].notna()]

# hash counts
hash_clique_counts = df_with_hash.groupby('hash')['clique'].nunique()
hashes_across_cliques = hash_clique_counts[hash_clique_counts > 1].index

# filter
df_filtered = df_with_hash[~df_with_hash['hash'].isin(hashes_across_cliques)].copy()
def pick_row(group):
    # prefer dvi == True
    dvi_true = group[group['dvi'] == True]
    if not dvi_true.empty:
        return dvi_true.iloc[[0]]  # keep first with dvi==True
    else:
        return group.iloc[[0]]  # keep first row
df_deduplicated_hashes = df_filtered.groupby('hash', group_keys=False).apply(pick_row)

# combine
df_deduplicated = pd.concat([df_deduplicated_hashes, df_no_hash], ignore_index=True)

print(f"After deduplication, dataset size: {len(df_deduplicated):,}")
print(f"Deduplication removed {len(df) - len(df_deduplicated):,} rows.")

df_deduplicated = df_deduplicated[df_deduplicated.groupby("clique")["clique"].transform("size") >= 2]
print(f"After singleton removal, dataset size: {len(df_deduplicated):,}")


/tmp/ipykernel_490100/1216921341.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_deduplicated_hashes = df_filtered.groupby('hash', group_keys=False).apply(pick_row)


After deduplication, dataset size: 1,349,125
Deduplication removed 40,650 rows.
After singleton removal, dataset size: 1,349,017


In [14]:
print_df_summary(df_deduplicated, "train")
print_df_summary(df_deduplicated, "valid")
print_df_summary(df_deduplicated, "test")


Summary statistics for train
clique       | Unique:  78,640 | Total: 1,007,643
version      | Unique: 1,007,643 | Total: 1,007,643
youtube_id   | Unique: 1,007,603 | Total: 1,007,643

DVI distribution:
  True:  328,117 (32.56%)
  False: 679,526 (67.44%)

Version per clique:
  Min:   2  | Max: 470  | Mean: 12.81  | Median:  5.00  | Std: 18.38
Summary statistics for valid
clique       | Unique:   8,736 | Total: 110,587
version      | Unique: 110,587 | Total: 110,587
youtube_id   | Unique: 110,586 | Total: 110,587

DVI distribution:
  True:   35,800 (32.37%)
  False:  74,787 (67.63%)

Version per clique:
  Min:   2  | Max: 266  | Mean: 12.66  | Median:  5.00  | Std: 17.94
Summary statistics for test
clique       | Unique:   9,795 | Total: 230,787
version      | Unique: 230,787 | Total: 230,787
youtube_id   | Unique: 230,785 | Total: 230,787

DVI distribution:
  True:  112,903 (48.92%)
  False: 117,884 (51.08%)

Version per clique:
  Min:   2  | Max: 650  | Mean: 23.56  | Median:  8.00  | 

In [ ]:
def save_df_as_torch(df, out_path, split_col="split"):
    """
    Transform dataframe into 'info' and 'split' dicts, then save as a torch file.

    Args:
        df : pandas.DataFrame
            Must have columns: ['clique', 'version', split_col, ...]
        out_path : str
            Path to save .pt file
        split_col : str
            Column name to drop from 'info' dict (default: "split")
    Returns:
        dict with keys: 'info', 'split'
    """
    # 1. Build unique key = clique:version
    index = df["clique"].astype(str) + ":" + df["version"].astype(str)

    # 2. Build info_dict: key = clique:version, value = row dict excluding split_col
    drop_cols = [split_col] if split_col in df.columns else []
    info_df = df.drop(columns=drop_cols)
    info_dict = {
        idx: dict(zip(info_df.columns, row))
        for idx, row in zip(index, info_df.itertuples(index=False, name=None))
    }

    # 3. Build split_dict: split -> {clique: [versions]}
    split_dict = {}
    if split_col in df.columns:
        # Only include rows where split_col is not NaN
        for split, sub_df in df.dropna(subset=[split_col]).groupby(split_col):
            # Each split maps to a dict of cliques -> list of versions
            clique_dict = sub_df.groupby("clique")["version"].apply(list).to_dict()
            split_dict[split] = clique_dict

    # 4. Save torch file
    data = {"info": info_dict, "split": split_dict}
    torch.save(data, out_path)
    print(f"Saved torch file with keys: {list(data.keys())} to {out_path}")
    return data

## Fingerprinting?

In [47]:
es.Chromaprinter()(audio)


'AQACgUmiVUoSJkkC_sWJ0xaykNFh4RcCcS_aB_l2_Gge9Ad_JMtphCeuV3i1EkyISoyKS0jtI7lESRHuHA3DDx3D4IqCI9kRZs0qXMfvBJ0sNFUYxFokIZkdNL8wM1EsNLkjtEmINVoU5Ej4DOdxg56eobsOJpOW4Am6X2h2I_ocQdP6IYzDYI-C60FlPQijbYf6oVQiUkEYPhJy7pDy5AhTh8erJcKXTBzaD0yiJRH4BK-Q7B-y5HiiD6eXoPqH5hOeIEfCXBo-XUGX9PDRRwlxHcmPNGbwzgu4jWnQ62iUMXgUIumRbg_OB73gPai0JEKuIXEOXwoehsJFXcX16PAYHeQkDcmDNLxwJtSE72iiMSFqZbiOZEOWVMWZD6-S4jW8S3iQ70bCGxfT4pIsNBSDSlIWXEeCVBNznJnxO3hMEs3BXNmRI1nyo5Se40qUC83RcQM_GfmR8MXF9Hjo4PVhJVXwDj8SJUd4PUJ1eDc6LUqO8wj_IVkcBmeOP8Wjjjmao-KyI0ey_CgzPai05ML2FN3E4AlCf0gWh8HL4mMlPCbhlAmeCEfCI1_oCF9mNNGmoyuDPUJ-JMzxEBeZBi8peD_eKFHxRg20HCUpCc-DMjqaUFK44FHw8UfCM8gnPGJKsGeCpmxT9BOeFDmRMDoznI_AHk2pHNWF80gWhMuS4aNxP-iOxksS_BOSH-nw8EH1o3kCsonwIznSuB0u5tif4i8aTzk64wgTDdrpIM_R5B2qD0eY9EI7pljTQ3eP5gr65Dg7CWViaPnx5sfVFOn24UkUPcEJV2GRVyShl0OeB5V4o3mOm8ih5SJCaXpQ5rDKEH2CE_mHhDFz4r3wKUyK-7AeBZ9y_EVyHmFGUUrAZtHxNuAJa0oQJEvFHO_wMU3wWOJQ_UZjRUieIO1x7QGbR3Dag3SU4zWSI32GS8TPxPgXULqi4lKGnDaSJSglS8G3zMK0KEbzRCVuJJ-RdsPr4T7Kk2gkNcdVRci